# Módulo 3 · Clase 6 (Práctica) — De las RNN a los LLMs en código
### Deep Learning · Laboratorio

**Objetivos.** Cada estudiante habrá:
1. Tokenizado texto y construido un vocabulario.
2. Entrenado una **LSTM desde cero** para clasificar sentimiento.
3. Hecho **fine-tuning de un modelo tipo BERT** con Hugging Face.
4. Usado un **LLM generativo** y explorado el efecto de la temperatura.

**Agenda (≈ 3 h):**
| Bloque | Tema | ~min |
|---|---|---|
| 0 | Setup y datos | 15 |
| 1 | Tokenización y vocabulario | 25 |
| 2 | Clasificador LSTM desde cero | 45 |
| — | *Descanso* | 10 |
| 3 | Fine-tuning de BERT (Hugging Face) | 45 |
| 4 | Un LLM generativo | 25 |
| 5 | Mini-reto | 15 |


In [ ]:
# Bloque 0 · Setup
!pip install -q transformers datasets
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, matplotlib.pyplot as plt
from torch.utils.data import DataLoader
torch.manual_seed(0); np.random.seed(0)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

### Dataset: `rotten_tomatoes`
Reseñas de cine etiquetadas como positivas (1) o negativas (0). Es pequeño y va rápido. Usamos un subconjunto para que el lab sea ágil.

In [ ]:
from datasets import load_dataset
ds = load_dataset("rotten_tomatoes")
# Subconjuntos para agilidad
train_data = ds["train"].shuffle(seed=0).select(range(3000))
test_data  = ds["test"].select(range(1000))
print(train_data[0])
print("train:", len(train_data), "| test:", len(test_data))

## Bloque 1 · Tokenización y vocabulario

Construyamos un tokenizador simple (por palabras) y un vocabulario a partir del train. Reservamos dos tokens especiales: `<pad>` (relleno) y `<unk>` (palabra desconocida).

In [ ]:
import re
from collections import Counter

def tokenizar(texto):
    return re.findall(r"[a-z']+", texto.lower())

# Construir vocabulario con las palabras más frecuentes
contador = Counter()
for ej in train_data: contador.update(tokenizar(ej["text"]))
vocab = {"<pad>":0, "<unk>":1}
for palabra, _ in contador.most_common(8000):
    vocab[palabra] = len(vocab)
print("tamaño del vocabulario:", len(vocab))
print("ejemplo:", tokenizar(train_data[0]["text"])[:8])

def codificar(texto, max_len=40):
    ids = [vocab.get(t, 1) for t in tokenizar(texto)][:max_len]
    return ids if ids else [1]

In [ ]:
# Collate: codifica un batch y lo rellena (padding) a la misma longitud
from torch.nn.utils.rnn import pad_sequence
def collate(batch):
    seqs = [torch.tensor(codificar(b["text"])) for b in batch]
    labels = torch.tensor([b["label"] for b in batch])
    lengths = torch.tensor([len(s) for s in seqs])
    padded = pad_sequence(seqs, batch_first=True, padding_value=0)
    return padded, lengths, labels

train_loader = DataLoader(train_data, batch_size=64, shuffle=True, collate_fn=collate)
test_loader  = DataLoader(test_data,  batch_size=128, collate_fn=collate)
xb, lb, yb = next(iter(train_loader))
print("batch:", xb.shape, "| longitudes:", lb[:5].tolist(), "| labels:", yb[:5].tolist())

## Bloque 2 · Clasificador LSTM desde cero

Arquitectura: **Embedding → LSTM → capa lineal**. Usamos el último estado oculto como resumen de la reseña para clasificar.

In [ ]:
class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim=100, hidden=128, n_classes=2):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm  = nn.LSTM(embed_dim, hidden, batch_first=True)
        self.fc    = nn.Linear(hidden, n_classes)
    def forward(self, x, lengths):
        e = self.embed(x)
        # empaquetar para que la LSTM ignore el padding
        packed = nn.utils.rnn.pack_padded_sequence(e, lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, (h_n, _) = self.lstm(packed)
        return self.fc(h_n[-1])           # último estado oculto -> clases

model = LSTMClassifier(len(vocab)).to(device)
print("parámetros:", sum(p.numel() for p in model.parameters()))

In [ ]:
# Entrenamiento
def evaluar(model, loader):
    model.eval(); correct=total=0
    with torch.no_grad():
        for x, lengths, y in loader:
            x, y = x.to(device), y.to(device)
            pred = model(x, lengths).argmax(1)
            correct += (pred==y).sum().item(); total += len(y)
    return correct/total

opt = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()
for epoch in range(5):
    model.train()
    for x, lengths, y in train_loader:
        x, y = x.to(device), y.to(device)
        opt.zero_grad()
        loss = loss_fn(model(x, lengths), y)
        loss.backward(); opt.step()
    print(f"época {epoch+1} | test acc: {evaluar(model, test_loader):.3f}")

### 🧩 Ejercicio 1
Prueba tu clasificador con frases tuyas. Escribe una función `predecir(texto)` que devuelva "positivo"/"negativo".

In [ ]:
# @title Solución
def predecir(texto):
    model.eval()
    x = torch.tensor([codificar(texto)]).to(device)
    lengths = torch.tensor([x.shape[1]])
    with torch.no_grad():
        prob = torch.softmax(model(x, lengths), dim=1)[0]
    return ("positivo" if prob.argmax().item()==1 else "negativo", round(prob.max().item(),2))

for t in ["a brilliant and moving masterpiece", "boring, dull and a complete waste of time"]:
    print(t, "->", predecir(t))

## Bloque 3 · Fine-tuning de BERT (Hugging Face)

En vez de entrenar desde cero, partimos de un Transformer **preentrenado** y lo adaptamos. Usamos **DistilBERT** (una versión más liviana y rápida de BERT). El propio modelo trae su **tokenizador** (sub-palabras), así que no construimos vocabulario a mano.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

ckpt = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(ckpt)

# Tokenización con el tokenizador del modelo
def tok_batch(batch):
    enc = tokenizer([b["text"] for b in batch], padding=True, truncation=True,
                    max_length=64, return_tensors="pt")
    enc["labels"] = torch.tensor([b["label"] for b in batch])
    return enc

bert_train = DataLoader(train_data, batch_size=32, shuffle=True, collate_fn=tok_batch)
bert_test  = DataLoader(test_data,  batch_size=64, collate_fn=tok_batch)

# Ejemplo de tokenización en sub-palabras
print(tokenizer.tokenize("an unforgettable cinematic experience"))

In [ ]:
# Cargar el modelo con una cabeza de clasificación (2 clases) y hacer fine-tuning
model_bert = AutoModelForSequenceClassification.from_pretrained(ckpt, num_labels=2).to(device)
opt = torch.optim.AdamW(model_bert.parameters(), lr=2e-5)

def evaluar_bert(model, loader):
    model.eval(); correct=total=0
    with torch.no_grad():
        for batch in loader:
            batch = {k:v.to(device) for k,v in batch.items()}
            logits = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"]).logits
            correct += (logits.argmax(1)==batch["labels"]).sum().item(); total += len(batch["labels"])
    return correct/total

for epoch in range(2):       # 2 épocas bastan al partir de un modelo preentrenado
    model_bert.train()
    for batch in bert_train:
        batch = {k:v.to(device) for k,v in batch.items()}
        opt.zero_grad()
        out = model_bert(**batch)        # al pasar labels, devuelve la loss
        out.loss.backward(); opt.step()
    print(f"época {epoch+1} | test acc: {evaluar_bert(model_bert, bert_test):.3f}")

Fíjate cómo DistilBERT supera a la LSTM con solo **2 épocas**: el preentrenamiento sobre enormes cantidades de texto ya le dio un fuerte conocimiento del lenguaje.

### 🧩 Ejercicio 2
Escribe `predecir_bert(texto)` usando el tokenizador y el modelo afinado, y pruébala con tus frases.

In [ ]:
# @title Solución
def predecir_bert(texto):
    model_bert.eval()
    enc = tokenizer(texto, return_tensors="pt", truncation=True, max_length=64).to(device)
    with torch.no_grad():
        prob = torch.softmax(model_bert(**enc).logits, dim=1)[0]
    return ("positivo" if prob.argmax().item()==1 else "negativo", round(prob.max().item(),2))

for t in ["a brilliant and moving masterpiece", "boring, dull and a complete waste of time"]:
    print(t, "->", predecir_bert(t))

## Bloque 4 · Un LLM generativo

Hasta ahora clasificamos (comprensión). Ahora **generamos** texto con un modelo autoregresivo solo-decoder. Usamos **DistilGPT-2** (pequeño) vía el `pipeline` de Hugging Face.

In [ ]:
from transformers import pipeline
generador = pipeline("text-generation", model="distilgpt2",
                     device=0 if device=="cuda" else -1)

prompt = "Deep learning is"
salida = generador(prompt, max_new_tokens=40, do_sample=False)   # greedy
print(salida[0]["generated_text"])

### El efecto de la temperatura
La **temperatura** controla cuán "creativo" es el muestreo: baja → conservador y repetitivo; alta → diverso y arriesgado. Compáralo.

In [ ]:
for temp in [0.3, 0.7, 1.2]:
    out = generador("Once upon a time", max_new_tokens=30, do_sample=True,
                    temperature=temp, top_k=50)
    print(f"\n--- temperatura {temp} ---")
    print(out[0]["generated_text"])

### 🧩 Ejercicio 3
Los LLMs generativos resuelven tareas **mediante prompting**, sin reentrenar. Diseña un prompt *few-shot* que induzca al modelo a clasificar sentimiento (dale 2-3 ejemplos en el prompt y luego una frase nueva). ¿Funciona con un modelo tan pequeño? Comenta las limitaciones.

In [ ]:
# @title Solución (ejemplo de prompt few-shot)
prompt = (
    "Review: I loved this film. Sentiment: positive\n"
    "Review: Terrible and boring. Sentiment: negative\n"
    "Review: A beautiful, moving story. Sentiment:"
)
print(generador(prompt, max_new_tokens=3, do_sample=False)[0]["generated_text"])
print("\nNota: distilgpt2 es muy pequeño; el few-shot suele fallar.")
print("Modelos grandes (GPT-4, Claude, Llama) hacen esto de forma fiable: es el poder de la escala.")

## Bloque 5 · 🏁 Mini-reto

Elige uno:
- **Mejora la LSTM:** bidireccional (`bidirectional=True`), más capas, dropout, embeddings preentrenados (GloVe). ¿Cuánta accuracy logras?
- **Fine-tuning eficiente:** aplica **LoRA** a DistilBERT con la librería `peft` y compara la accuracy y el número de parámetros entrenables contra el fine-tuning completo.

```python
# Pista para LoRA:
# !pip install peft
# from peft import LoraConfig, get_peft_model
# config = LoraConfig(task_type="SEQ_CLS", r=8, lora_alpha=16, target_modules=["q_lin","v_lin"])
# model_lora = get_peft_model(model_bert_base, config)
# model_lora.print_trainable_parameters()
```

In [ ]:
# TODO: tu solución
...

## Cierre

Recorriste el NLP práctico de punta a punta: tokenización, una LSTM desde cero, fine-tuning de un Transformer preentrenado y generación con un LLM.

**En el Módulo 4** llegamos a los temas avanzados: modelos generativos, multimodales y agentes.